# Evaluate CLACELL Package

In [1]:
# Scumi annotated labels

import anndata as ad
import scanpy as sc
# For saving results on HPC Cluster
import joblib
import pandas as pd
import os

# Load training data
#adata = ad.io.read_h5ad('../data/CellTypistDataset/global_annotated.h5ad')
adata = ad.io.read_h5ad('../data/CellTypistDataset/CountAdded_PIP_global_object_for_cellxgene_annotated.h5ad')

# Filter blood data
adata = adata[adata.obs['Organ'] == 'BLD'].copy()
print(adata)

# Use raw data instead of already preprocessed data
adata.X = adata.layers['counts'].copy()

# Preprocessing

# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

# Remove mitochondrial, ribosomal and hemoglobin
adata = adata[:, ~adata.var["mt"]].copy()
adata = adata[:, ~adata.var["ribo"]].copy()
adata = adata[:, ~adata.var["hb"]].copy()

# Doublet Detection
sc.pp.scrublet(adata, batch_key="Donor")
adata = adata[adata.obs['predicted_doublet'] == False].copy()


# Normalization

# Saving count data
adata.layers["counts"] = adata.X.copy()

# Normalizing to median total counts
sc.pp.normalize_total(adata, target_sum=1e4)
# Logarithmize the data
sc.pp.log1p(adata)

# Filtering Highly variable genes
print('Before filtering highly variable genes ---')
print(adata)

sc.pp.highly_variable_genes(adata, n_top_genes=10000)

# Apply filter
adata = adata[:, adata.var['highly_variable']].copy()

print('After filtering highly variable genes ---')
print(adata)

# Create train test split

# All Donors: ['621B', '637C', 'A35', 'A36', 'D496', 'D503']
donor_train = ['637C', 'A35', 'A36', 'D503']
donor_test = ['621B', 'D496']

adata_train = adata[
    adata.obs["Donor"].isin(donor_train)
].copy()

adata_test = adata[
    adata.obs["Donor"].isin(donor_test)
].copy()

# Check split
print(adata_train.obs['Donor'].unique())
print(adata_test.obs['Donor'].unique())

# Prepare Data for training
X_train = adata_train.to_df()
gene_names_train = adata_train.var_names
y_train = adata_train.obs['scumi-annotation']

X_test = adata_test.to_df()
gene_names_test = adata_test.var_names
y_test = adata_test.obs['scumi-annotation']

AnnData object with n_obs × n_vars = 27620 × 36473
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype', 'Sex', 'Age_range', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'scumi-annotation'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'Age_range_colors', 'Sex_colors', 'scrublet'
    obsm: 'X_umap'
    layers: 'counts'
Before filtering highly variable genes 

In [2]:
from sklearn.metrics import f1_score
import sys

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from clacell import BloodCellClassifier

classifier = BloodCellClassifier()

# 1. GridSearch
print("\n1. grid_search")
classifier.grid_search(X_train, y_train, X_test, y_test, n_jobs=3)

# 2. Train
print("\n2. train")
classifier.train(X_train, y_train, C=0.001)

# 3. Evaluate
print("\n3. evaluate")
classifier.evaluate(X_test, y_test)

# 4. Predict
print("\n4. predict")
predictions = classifier.predict(X_test)
print(f"Macro F1: {f1_score(y_test, predictions, average="macro")}")


1. grid_search
Fitting 3 folds for each of 5 candidates, totalling 15 fits


/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/li

Best parameters found: {'C': np.float64(0.0038062828501796006), 'class_weight': None, 'l1_ratio': 0.75, 'max_iter': 1000, 'solver': 'saga', 'tol': np.float64(0.012129626999995879)}
Evaluate model on test data...
--- In distribution testset ---
Baseline accuracy score 0.8827

                     precision    recall  f1-score   support

             B cell       0.87      0.88      0.88       120
     CD14+ monocyte       0.99      1.00      0.99      2575
        CD4+ T cell       0.89      0.99      0.94      3910
   Cytotoxic T cell       0.95      0.48      0.64      1824
     Dendritic cell       0.00      0.00      0.00         5
      Megakaryocyte       1.00      1.00      1.00         7
Natural killer cell       0.59      0.95      0.72       791
        Plasma cell       0.00      0.00      0.00        49

           accuracy                           0.88      9281
          macro avg       0.66      0.66      0.65      9281
       weighted avg       0.90      0.88      0.87 

/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` 

Random dropout accuracy score 0.8571
Total samples: 9281
Number of inconsistent predictions: 0
--- Out of data distribution ---
No out-of-distribution dataset path provided. Skipping out-of-distribution tests.

Start final training with best parameters on complete training data...
Train models with parameters: {'C': np.float64(0.0038062828501796006), 'class_weight': None, 'l1_ratio': 0.75, 'max_iter': 1000, 'solver': 'saga', 'tol': np.float64(0.012129626999995879)}

2. train
Train models with parameters: {'C': 0.001}

3. evaluate
Evaluate model on test data...
--- In distribution testset ---
Baseline accuracy score 0.9307

                     precision    recall  f1-score   support

             B cell       1.00      0.98      0.99       120
     CD14+ monocyte       1.00      1.00      1.00      2575
        CD4+ T cell       0.89      1.00      0.94      3910
   Cytotoxic T cell       0.99      0.67      0.80      1824
     Dendritic cell       1.00      0.40      0.57         5
  

In [12]:
# Test split on one dataframe
from sklearn.metrics import f1_score
import sys

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from clacell import BloodCellClassifier

classifier = BloodCellClassifier()

X = adata.to_df()
y = adata.obs['scumi-annotation']

# 1. GridSearch
print("\n1. grid_search")
classifier.grid_search(X, y, n_jobs=3)


1. grid_search
Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best parameters found: {'C': np.float64(1.2038063109020845), 'class_weight': None, 'l1_ratio': 1.0, 'max_iter': 1000, 'solver': 'saga', 'tol': np.float64(0.037466071161645416)}
Evaluate model on test data...
--- In distribution testset ---
Baseline accuracy score 0.9923

                     precision    recall  f1-score   support

             B cell       1.00      0.99      0.99        98
     CD14+ monocyte       1.00      1.00      1.00      1606
        CD4+ T cell       0.99      0.99      0.99      3346
   Cytotoxic T cell       0.98      0.98      0.98       734
     Dendritic cell       0.95      0.90      0.93        41
      Megakaryocyte       0.97      1.00      0.99        37
Natural killer cell       0.99      0.99      0.99      1008
        Plasma cell       1.00      0.96      0.98        24

           accuracy                           0.99      6894
          macro avg       0.98      0.98

In [10]:
import anndata as ad
# Load training data
#adata = ad.io.read_h5ad('../data/CellTypistDataset/global_annotated.h5ad')
adata_unprocessed = ad.io.read_h5ad('../data/CellTypistDataset/CountAdded_PIP_global_object_for_cellxgene_annotated.h5ad')

# Filter blood data
adata_unprocessed = adata_unprocessed[adata_unprocessed.obs['Organ'] == 'BLD'].copy()
print(adata_unprocessed)

# Use raw data instead of already preprocessed data
adata_unprocessed.X = adata_unprocessed.layers['counts'].copy()

AnnData object with n_obs × n_vars = 27620 × 36473
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype', 'Sex', 'Age_range', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'scumi-annotation'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'Age_range_colors', 'Sex_colors', 'scrublet'
    obsm: 'X_umap'
    layers: 'counts'


In [ ]:
from sklearn.metrics import f1_score
import sys
import os

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from clacell import BloodCellClassifier

classifier = BloodCellClassifier()

# 1. GridSearch
print("\n1. grid_search")
classifier.grid_search(adata_unprocessed, n_jobs=3)

# 2. Train
print("\n2. train")
classifier.train(adata_unprocessed, C=0.001)

# 3. Evaluate
print("\n3. evaluate")
classifier.evaluate(adata_unprocessed)

# 4. Predict
print("\n4. predict")
predictions = classifier.predict(adata_unprocessed)
print(f"Macro F1: {f1_score(adata.obs['scumi-annotation'], predictions, average="macro")}")